# Detectron2 Faster R-CNN Workflow

This is the standard workflow for training and evaluating a Faster R-CNN model using Detectron2.

```text
1. Install
        ↓
2. Import
        ↓
3. Download Dataset
        ↓
4. Register Dataset
        ↓
5. Visualize Dataset
        ↓
6. Configure Model
        ↓
7. Train
        ↓
8. Load Trained Model
        ↓
9. Predict
        ↓
10. Evaluate
```

## Workflow Explanation

### 1. Install
Install Detectron2 and all required dependencies.

### 2. Import
Import Detectron2 modules and common Python libraries.

### 3. Download Dataset
Download the dataset (e.g., from Roboflow) in COCO format.

### 4. Register Dataset
Register the training and testing datasets using `register_coco_instances()`.

### 5. Visualize Dataset
Visualize images and annotations to verify that the dataset has been loaded correctly.

### 6. Configure Model
Load the Faster R-CNN configuration and set hyperparameters such as:
- Learning Rate
- Batch Size
- Number of Classes
- Maximum Iterations

### 7. Train
Train the Faster R-CNN model on the custom dataset.

### 8. Load Trained Model
Load the trained weight file (`model_final.pth`) for inference.

### 9. Predict
Run inference on test images and visualize predicted bounding boxes.

### 10. Evaluate
Evaluate the trained model using COCO evaluation metrics such as mAP.

# My Detectron2 Faster R-CNN Notebook Workflow

This document maps each notebook cell to the standard Detectron2 Faster R-CNN workflow.

| Workflow | Notebook Cell | Status |
|----------|---------------|--------|
| ✅ Install Detectron2 | Cell 1 | ✔️ |
| ✅ Import Libraries | Cell 2 | ✔️ |
| ✅ Download Dataset (Roboflow) | Cell 4 | ✔️ |
| ✅ Register Dataset | Cell 5 | ✔️ |
| ✅ Visualize Dataset | Cell 6 | ✔️ |
| ✅ Configure Model | Cell 10 (`cfg = get_cfg()`) | ✔️ |
| ✅ Train Model | Cell 10 (`trainer.train()`) | ✔️ |
| ✅ Load Trained Model | Cell 15 | ✔️ |
| ✅ Predict | Cell 16 | ✔️ |
| ✅ Evaluate | Cell 18 | ✔️ |

---

## Overall Pipeline

```text
Install Detectron2
        │
        ▼
Import Libraries
        │
        ▼
Download Dataset (Roboflow)
        │
        ▼
Register Dataset
        │
        ▼
Visualize Dataset
        │
        ▼
Configure Faster R-CNN
        │
        ▼
Train Model
        │
        ▼
Load model_final.pth
        │
        ▼
Predict on Test Images
        │
        ▼
Evaluate (COCO Metrics)
```

## Summary

The notebook follows the standard Detectron2 Faster R-CNN pipeline:

1. Install Detectron2
2. Import required libraries
3. Download the dataset
4. Register the dataset
5. Visualize annotations
6. Configure the Faster R-CNN model
7. Train the model
8. Load the trained weights
9. Run inference on test images
10. Evaluate model performance

### Detectron2 Depecdencies Install

In [ ]:
!pip install pyyaml==5.1

import torch
TORCE_VERSON = " . ".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[1]
print("torch : ", TORCE_VERSON, ", cuda : ", CUDA_VERSION)

!python -m pip install 'git+https://github.com/facebookresearch/detectron2'


### Import Libraries

In [ ]:
# Setup Detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# Import common libraries
import numpy as np
import os
import json
import cv2
import random

from google.colab.patches import cv2_imshow

# Import Detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

### Roboflow Download

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="trta6GNa9BxIphg4DPfG")
project = rf.workspace("abdullahs-workspace-rcwes").project("person-detection-9a6mk-9in4p")
version = project.version(2)
dataset = version.download("coco")

### Dataset Registration

In [ ]:
from detectron2.data.datasets import register_coco_instances

register_coco_instances(
    "Person_Train_data",
    {},
    "/content/Person-detection-2/train/_annotations.coco.json",
    "/content/Person-detection-2/train"
)

register_coco_instances(
    "Person_Test_data",
    {},
    "/content/Person-detection-2/test/_annotations.coco.json",
    "/content/Person-detection-2/test"
)

### Visualization

In [ ]:
# Visualize train data

my_dataset_train_metadata = MetadataCatalog.get("Person_Train_data")
dataset_dicts = DatasetCatalog.get("Person_Train_data")

import random
from detectron2.utils.visualizer import Visualizer

for d in random.sample(dataset_dicts, 3):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(
        img[:, :, ::-1],
        metadata=my_dataset_train_metadata,
        scale=1
    )
    vis = visualizer.draw_dataset_dict(d)
    cv2_imshow(vis.get_image()[:, :, ::-1])

In [ ]:
# import os

# print(os.listdir("/content"))

In [ ]:
# import os

# print(os.listdir("/content/Person-detection-2/train"))

### Training

In [ ]:
from detectron2.engine import DefaultTrainer

cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
    )
)

# Train dataset
cfg.DATASETS.TRAIN = ("Person_Train_data",)
cfg.DATASETS.TEST = ()

cfg.DATALOADER.NUM_WORKERS = 2

# COCO pretrained weight load
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
)

cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.0025
cfg.SOLVER.MAX_ITER = 300

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)

trainer.train()  # Training start

### Training graph shows

In [ ]:
# look at traning curve in tensorboard

%load_ext tensorboard
%tensorboard --logdir output

### load model


In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5 # set the testing threshold for this model
cfg.DATASETS.TEST = ("Person_Test_data",)
predictor = DefaultPredictor(cfg)

### Inference with Detectron2 Saved Weights

In [40]:
# class information load of datasets
test_metadata = MetadataCatalog.get("Person_Test_data")

### Prediction on Test Images

In [ ]:
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from google.colab.patches import cv2_imshow
import glob
import cv2

test_metadata = MetadataCatalog.get("Person_Test_data")

for imageName in glob.glob("/content/Person-detection-2/test/*.jpg"):

    im = cv2.imread(imageName)

    outputs = predictor(im)

    v = Visualizer(
        im[:, :, ::-1],
        metadata=test_metadata,
        scale=0.8
    )

    out = v.draw_instance_predictions(
        outputs["instances"].to("cpu")
    )

    cv2_imshow(out.get_image()[:, :, ::-1])

### Evaluation

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator = COCOEvaluator("Person_Test_data", output_dir = "./output")
val_loader = build_detection_test_loader(cfg, "Person_Test_data")
print(inference_on_dataset(predictor.model, val_loader, evaluator))

In [ ]:
# save

f = open('config.yml', 'w')
f.write(cfg.dump())
f.close()